# 01 · Comprensión de los datos (EDA) con Dask
**Proyecto Big Data · Amazon Books Reviews · Metodología CRISP-DM (fases 1 y 2)**

| | |
|---|---|
| **Fuente** | Kaggle – *Amazon Books Reviews* (mohamedbakhet), derivado de los datos de reseñas de Amazon (McAuley, UCSD) |
| **Archivos** | `Books_rating.csv` (reseñas) y `books_data.csv` (metadatos de libros de Google Books) |
| **Objetivo de análisis** | Identificar qué características del libro y de la reseña se asocian con la **insatisfacción del lector**, para orientar decisiones de catálogo y promoción |
| **Unidad de análisis** | La **reseña** individual (un usuario califica un libro en un momento dado) |
| **Variable objetivo** | `is_negative` = 1 si `review/score` ≤ 2 |

> Requisito previo: haber ejecutado `python -m src.convert` (o el flujo de Prefect) para tener la capa *bronze* en Parquet.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from src.utils import load_config

cfg = load_config()
FIG = cfg["paths"]["reports_dir"] / "figures"; FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.float_format", "{:,.3f}".format)

def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIG / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()

## 1. Cluster de Dask
Se usa el perfil **B** (4 workers × 1 hilo, 1.5 GiB por worker). El dashboard (enlace abajo) sirve como evidencia: tomen captura de *Task Stream* y *Workers Memory* mientras corre el notebook.

In [ ]:
p = cfg["dask"]["profiles"]["B"]
cluster = LocalCluster(n_workers=p["n_workers"], threads_per_worker=p["threads_per_worker"],
                       memory_limit=p["memory_limit"])
client = Client(cluster)
client

## 2. Tamaño del dataset y carga

In [ ]:
raw = cfg["paths"]["raw_dir"]
tam = pd.DataFrame([{"archivo": f.name, "tamano_GB": f.stat().st_size / 1e9} for f in sorted(raw.glob("*.csv"))])
display(tam); print(f"Total: {tam.tamano_GB.sum():.2f} GB")

bronze = cfg["paths"]["bronze_dir"]
cols = ["book_id", "title", "price", "user_id", "profile_name", "helpfulness", "score",
        "review_time", "summary", "text_len", "text_words"]
rev = dd.read_parquet(bronze / "reviews", columns=cols)
books = dd.read_parquet(bronze / "books")
print("Particiones reseñas:", rev.npartitions)

In [ ]:
n_rev, n_books = rev.shape[0].compute(), books.shape[0].compute()
mem = rev.memory_usage(deep=True).sum().compute() / 1e9
print(f"Reseñas: {n_rev:,} filas x {rev.shape[1]} columnas | memoria (sin texto completo): {mem:.2f} GB")
print(f"Libros:  {n_books:,} filas x {books.shape[1]} columnas")
rev.head()

## 3. Tipos de datos

In [ ]:
tipos = pd.DataFrame({"dtype": rev.dtypes.astype(str)})
tipos["tipo_analitico"] = ["identificador", "texto/categórica", "numérica continua", "identificador",
                           "texto", "texto 'x/y' (requiere parseo)", "numérica ordinal (1-5)",
                           "fecha (Unix)", "texto corto", "numérica discreta", "numérica discreta"]
tipos

## 4. Valores faltantes

In [ ]:
nulos = (rev.isna().mean().compute() * 100).sort_values(ascending=False).rename("% nulos reseñas")
nulos_b = (books.isna().mean().compute() * 100).sort_values(ascending=False).rename("% nulos libros")
display(nulos.to_frame()); display(nulos_b.to_frame())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
nulos.plot.barh(ax=ax[0], color="#4C72B0"); ax[0].set_title("Reseñas: % de nulos")
nulos_b.plot.barh(ax=ax[1], color="#DD8452"); ax[1].set_title("Libros: % de nulos")
guardar("01_nulos")

## 5. Medidas de tendencia central y dispersión

In [ ]:
num = rev[["score", "price", "text_len", "text_words"]]
desc = num.describe().compute().T
desc["mediana"] = num.quantile(0.5).compute()
desc["moda"] = [num[c].value_counts().compute().idxmax() if num[c].notnull().any().compute() else np.nan for c in num.columns]
desc["asimetria"] = num.sample(frac=min(1, 500_000 / n_rev), random_state=1).compute().skew()
desc

## 6. Distribución del puntaje (score)

In [ ]:
dist = rev["score"].value_counts().compute().sort_index()
pct = dist / dist.sum() * 100
display(pd.DataFrame({"reseñas": dist, "%": pct}))
ax = pct.plot.bar(color=["#C44E52", "#DD8452", "#CCB974", "#55A868", "#4C72B0"], figsize=(7, 4))
ax.set_title("Distribución de puntajes"); ax.set_ylabel("% de reseñas"); ax.set_xlabel("Estrellas")
for i, v in enumerate(pct): ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
guardar("02_distribucion_score")

## 7. Utilidad de la reseña (votos *helpful*)

In [ ]:
# Se reutiliza la MISMA función del pipeline (src/transform.py), aplicada por partición
from src.transform import add_review_features
thr = cfg["analysis"]["negative_threshold"]
fe = rev.map_partitions(add_review_features, thr)   # parsea 'x/y', fecha, objetivo, título normalizado
print("% de reseñas sin ningún voto:", round((fe["votes_total"] == 0).mean().compute() * 100, 2))
util = fe.groupby("score").agg({"helpful_ratio": "mean", "votes_total": "mean"}).compute()
display(util)
util["helpful_ratio"].plot.bar(color="#55A868", figsize=(7, 4), title="Proporción de votos 'útil' según el puntaje")
guardar("03_utilidad_por_score")

## 8. Comportamiento en el tiempo

In [ ]:
anual = fe.groupby("review_year").agg({"score": ["count", "mean"]}).compute()
anual.columns = ["reseñas", "puntaje_promedio"]; anual = anual.sort_index()
display(anual.tail(10))
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(anual.index, anual["reseñas"], color="#4C72B0", alpha=.7); ax1.set_ylabel("Reseñas")
ax2 = ax1.twinx(); ax2.plot(anual.index, anual["puntaje_promedio"], color="#C44E52", marker="o")
ax2.set_ylabel("Puntaje promedio"); ax1.set_title("Volumen de reseñas y puntaje promedio por año")
guardar("04_tendencia_anual")

## 9. Longitud del texto

In [ ]:
palabras = fe.groupby("score")["text_words"].mean().compute()
display(palabras.rename("palabras promedio"))
muestra = rev[["score", "text_words"]].sample(frac=min(1, 300_000 / n_rev), random_state=1).compute()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(np.log10(muestra["text_words"].clip(lower=1)), bins=50, ax=ax[0])
ax[0].set_title("Distribución de palabras (escala log10)"); ax[0].set_xlabel("log10(palabras)")
sns.boxplot(data=muestra, x="score", y="text_words", showfliers=False, ax=ax[1])
ax[1].set_title("Palabras por puntaje (sin atípicos)")
guardar("05_longitud_texto")

## 10. Usuarios: concentración de la actividad

In [ ]:
por_usuario = rev.groupby("user_id")["score"].count().compute().sort_values(ascending=False)
print(f"Usuarios únicos: {len(por_usuario):,} | reseñas anónimas: {rev['user_id'].isna().mean().compute()*100:.1f} %")
top1 = por_usuario.head(max(1, len(por_usuario) // 100)).sum() / por_usuario.sum() * 100
print(f"El 1 % de usuarios más activos escribe el {top1:.1f} % de las reseñas")
display(por_usuario.describe())
ax = por_usuario.clip(upper=50).plot.hist(bins=50, log=True, figsize=(7, 4), color="#8172B2")
ax.set_title("Reseñas por usuario (recortado en 50, escala log)")
guardar("06_reseñas_por_usuario")

## 11. Libros: popularidad y cola larga

In [ ]:
por_libro = fe.groupby("title_norm")["score"].agg(["count", "mean"]).compute().sort_values("count", ascending=False)
print(f"Títulos únicos: {len(por_libro):,}")
acum = por_libro["count"].cumsum() / por_libro["count"].sum() * 100
pareto = (acum < 80).mean() * 100
print(f"El {pareto:.1f} % de los títulos concentra el 80 % de las reseñas")
display(por_libro.head(10))
ax = acum.reset_index(drop=True).plot(figsize=(7, 4), color="#C44E52")
ax.axhline(80, ls="--", c="gray"); ax.set_title("Curva de Pareto: % acumulado de reseñas por título")
ax.set_xlabel("Títulos ordenados por popularidad"); guardar("07_pareto_libros")

## 12. Metadatos de libros y cobertura del cruce

In [ ]:
from src.transform import clean_books
b = clean_books(books.compute())
print(f"Libros con metadatos (únicos): {len(b):,}")
cobertura = fe["title_norm"].isin(b["title_norm"].tolist()).mean().compute() * 100
print(f"% de reseñas cuyo libro tiene metadatos: {cobertura:.1f} %")
cats = b["main_category"].value_counts().head(15)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cats.sort_values().plot.barh(ax=ax[0], color="#4C72B0"); ax[0].set_title("Top 15 categorías (libros)")
b["published_year"].dropna().astype(int).clip(1900, 2015).plot.hist(bins=40, ax=ax[1], color="#DD8452")
ax[1].set_title("Año de publicación"); guardar("08_metadatos_libros")

## 13. Duplicados y calidad

In [ ]:
clave = ["user_id", "title_norm", "review_time", "score", "summary"]
n_unicos = fe[clave].drop_duplicates().shape[0].compute()
print(f"Filas duplicadas: {n_rev - n_unicos:,} ({(n_rev - n_unicos) / n_rev * 100:.2f} %)")
print(f"Reseñas sin puntaje: {rev['score'].isna().sum().compute():,}")
print(f"Precio disponible en {rev['price'].notnull().mean().compute()*100:.1f} % de las reseñas -> no se usa como variable principal")

## 14. Construcción de la variable objetivo

In [ ]:
# is_negative ya fue construida por add_review_features: score <= umbral (config.yaml)
balance = fe["is_negative"].value_counts(normalize=True).compute() * 100
display(balance.rename("% de reseñas").to_frame())
print("Clases desbalanceadas -> en el modelo se usa class_weight='balanced' y se reporta PR AUC además de ROC AUC.")

## 15. Relación de las características candidatas con la variable objetivo

In [ ]:
cand = fe[["is_negative", "text_words", "votes_total", "helpful_ratio", "review_year", "price", "summary_len", "summary_exclam"]]
m = cand.sample(frac=min(1, 400_000 / n_rev), random_state=7).compute().astype("float64")
corr = m.corr(method="spearman")
plt.figure(figsize=(7, 5)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlación de Spearman (muestra)"); guardar("09_correlaciones")
corr["is_negative"].drop("is_negative").sort_values()

## 16. Hallazgos preliminares (completar con los resultados reales)

1. **Sesgo positivo:** la mayoría de reseñas son de 4–5 estrellas; las negativas (≤ 2) son minoría → problema desbalanceado.
2. **Utilidad:** comparar la proporción de votos útiles entre reseñas negativas y positivas.
3. **Longitud:** ¿las reseñas negativas son más largas?
4. **Concentración:** pocos títulos y pocos usuarios concentran gran parte de las reseñas (Pareto).
5. **Calidad:** precio casi vacío, reseñas anónimas, títulos sin metadatos y duplicados entre ediciones.

### ¿Se requiere información adicional de contexto? (punto c)
* **Cómo se generó el dato:** el sistema de votos *helpful* de Amazon y el cambio de interfaz a lo largo de los años afectan la comparabilidad temporal.
* **Taxonomía de categorías:** proviene de Google Books (no de Amazon) y solo se toma la primera categoría.
* **Precio y formato:** el precio está casi vacío; datos de precio/formato (Kindle vs. físico) enriquecerían el análisis.
* **Eventos externos:** el lanzamiento del Kindle (2007) y el crecimiento de Amazon explican el aumento de volumen.
* **Inconsistencia de la fuente:** la descripción de Kaggle menciona Goodreads, pero el contenido es de Amazon.

In [ ]:
client.close(); cluster.close()